# Arrays and Vectorized Computing

Every numerical method in the rest of this book is expressed in terms of arrays.
This chapter is where that habit is built: how to represent a physical quantity
over a range of conditions as a NumPy array, why that is faster than a loop, and
how broadcasting lets one expression cover many cases at once.

The running example is **vapor pressure**, which we will need constantly later
when we solve flash calculations and distillation columns.

## Learning objectives

By the end of this chapter you should be able to:

- write a Python function with a docstring, and unpack arguments from a tuple
- represent a physical quantity over a range of conditions as a NumPy array, and
  explain why that is faster than a loop
- use broadcasting to evaluate a correlation for several species at once
- produce a labeled, publication-quality figure

## Setup

Every chapter begins with this cell. `use_course_style` applies the book-wide
Matplotlib settings so figures are consistent throughout — see
`cheme_numerics/plotting.py` for exactly what it changes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cheme_numerics import use_course_style

use_course_style()

## Functions and the Antoine equation

The Antoine equation correlates the saturation pressure of a pure species with
temperature:

$$
\log_{10} P^{\mathrm{sat}} = A - \frac{B}{C + T}
$$

where $P^{\mathrm{sat}}$ is in mmHg and $T$ in $^\circ$C, and $A$, $B$, $C$ are
species-specific constants fitted to experimental data. It is empirical: outside
the temperature range it was fitted over, it is not merely inaccurate but
qualitatively wrong.

Here is that equation as a Python function.

In [ ]:
def antoine_psat(T, A, B, C):
    """Saturation pressure from the Antoine equation.

    Parameters
    ----------
    T : float or array_like
        Temperature in degrees Celsius.
    A, B, C : float
        Antoine constants for the species, consistent with pressure in mmHg
        and temperature in degrees Celsius.

    Returns
    -------
    float or ndarray
        Saturation pressure in mmHg, broadcast to the shape of the inputs.
    """
    return 10.0 ** (A - B / (C + T))


# Antoine constants: benzene valid ~8-103 C, toluene ~6-137 C.
BENZENE = (6.90565, 1211.033, 220.790)
TOLUENE = (6.95464, 1344.800, 219.482)

antoine_psat(80.0, *BENZENE)

Three things worth noticing:

- The triple-quoted string under `def` is a **docstring**. Try `antoine_psat?`
  in a cell — Jupyter renders it as help. Write one for every function you define
  in the homework.
- `*BENZENE` **unpacks** the tuple into the arguments `A`, `B`, `C`. It is
  equivalent to typing all three numbers out, but harder to get wrong.
- The function never checks that `T` is in the valid range. That is a deliberate
  simplification, and Exercise 0.2 asks you to confront it.

## Arrays instead of loops

Suppose we want $P^{\mathrm{sat}}$ at many temperatures. The instinct from most
introductory courses is a loop:

In [ ]:
T_list = [20.0 + 0.4 * i for i in range(200)]

psat_list = []
for T_i in T_list:
    psat_list.append(antoine_psat(T_i, *BENZENE))

psat_list[:4]

The NumPy way is to make the temperature an **array** and call the function once.
`antoine_psat` needs no modification: the arithmetic inside it already works
element-by-element on arrays.

In [ ]:
T = np.linspace(20.0, 100.0, 200)      # 200 points, both endpoints included
psat_benzene = antoine_psat(T, *BENZENE)

print(type(psat_benzene).__name__, psat_benzene.shape)
print(psat_benzene[:4])

This is not only shorter — it is substantially faster, because the loop runs in
compiled C rather than in the Python interpreter. Let's measure it.

In [ ]:
T_big = np.linspace(20.0, 100.0, 100_000)

loop = %timeit -o -q -n 3 -r 3 [antoine_psat(t, *BENZENE) for t in T_big]
vec  = %timeit -o -q -n 3 -r 3 antoine_psat(T_big, *BENZENE)

print(f"loop:       {loop.average * 1e3:8.2f} ms")
print(f"vectorized: {vec.average * 1e3:8.2f} ms")
print(f"speedup:    {loop.average / vec.average:8.0f}x")

:::{important}
That speedup is why every numerical method in this book is written in terms of
array operations. When a calculation feels slow, the first question is almost
always "where is the Python-level loop?" — not "do I need a faster computer?"
:::

## Broadcasting: several species at once

We can go further and evaluate both species in a single call. Stack the Antoine
constants into arrays of shape `(2,)` and reshape the temperature to `(200, 1)`;
NumPy then pairs every temperature with every species:

$$
\underbrace{(200,\,1)}_{\text{temperature}}
\;\text{with}\;
\underbrace{(2,)}_{\text{species}}
\;\longrightarrow\;
\underbrace{(200,\,2)}_{\text{result}}
$$

This is **broadcasting**. It is the single most useful NumPy idea and a common
source of silent bugs, so check shapes as you go.

In [ ]:
A, B, C = np.array([BENZENE, TOLUENE]).T    # each of shape (2,)
print("constants:", A.shape, B.shape, C.shape)

psat = antoine_psat(T[:, None], A, B, C)    # T[:, None] has shape (200, 1)
print("result:   ", psat.shape)

:::{warning}
Had we written `antoine_psat(T, A, B, C)` with `T` of shape `(200,)` against
constants of shape `(2,)`, NumPy would have raised a shape-mismatch error — which
is the *good* outcome. The dangerous case is when two mismatched shapes happen to
be broadcast-compatible and you silently get an array of the wrong meaning. Print
`.shape` when you are unsure.
:::

## Plotting

Note the axis labels with units, the legend, and the horizontal line at
atmospheric pressure: where each curve crosses 760 mmHg is that species' normal
boiling point.

In [ ]:
fig, ax = plt.subplots()

ax.plot(T, psat[:, 0], label="benzene")
ax.plot(T, psat[:, 1], label="toluene")
ax.axhline(760.0, color="0.5", ls="--", lw=1.2)
ax.text(22.0, 800.0, "1 atm", color="0.4", fontsize=9)

ax.set_xlabel(r"temperature  $T$  [$^\circ$C]")
ax.set_ylabel(r"saturation pressure  $P^{\mathrm{sat}}$  [mmHg]")
ax.set_title("Antoine correlation for the benzene–toluene system")
ax.set_xlim(T[0], T[-1])
ax.set_ylim(0.0, 1600.0)
ax.legend()

plt.show()

Benzene is the more volatile species at every temperature shown — its curve lies
above toluene's throughout. That is what makes the pair separable by
distillation, and the vertical gap between the curves is, loosely, what sets how
easy the separation is.

Reading a boiling point off a graph is unsatisfying, though. We want the $T$ that
solves

$$
P^{\mathrm{sat}}(T) - 760 = 0,
$$

which is a **root-finding problem**. Notice what just happened: a physical
question became an equation of the form $f(T) = 0$. Much of this course consists
of recognizing that transformation, then choosing a method to finish the job.

## Exercises

:::{admonition} Exercise 0.1 — Unit conversion
:class: tip

Write `antoine_psat_bar(T_K, A, B, C)` that accepts temperature in **kelvin** and
returns pressure in **bar**, reusing `antoine_psat` rather than rewriting the
correlation. Verify that benzene at 353.25 K gives approximately 1.013 bar.
:::

:::{admonition} Exercise 0.2 — Extrapolation
:class: tip

The benzene constants above were fitted over roughly 8–103 $^\circ$C. Evaluate
the correlation at $T = 400\ ^\circ$C and compare with benzene's critical
pressure of 48.9 bar. Benzene's critical temperature is 289 $^\circ$C — what
should $P^{\mathrm{sat}}$ even *mean* above it? What does this tell you about
using empirical correlations outside their fitted range?
:::

:::{admonition} Exercise 0.3 — Broadcasting
:class: tip

Add $n$-hexane ($A = 6.87776$, $B = 1171.530$, $C = 224.366$) and regenerate the
figure with all three curves using a single call to `antoine_psat`. Your code
should not contain the literal `3` anywhere.
:::

:::{admonition} Exercise 0.4 — Reading the docs
:class: tip

`np.linspace(20, 100, 200)` includes both endpoints. Find the NumPy function that
generates evenly spaced values *excluding* the endpoint, and the `linspace`
argument that does the same. When would the difference matter in a numerical
method?
:::

## Further reading

Chapra and Canale {cite}`chapra2010numerical` is the standard reference for the
classical methods in this course. For chemical-engineering-specific treatments
with worked Python, Kantor's open-access notes {cite}`kantor2022` are excellent
and freely available.